# Notebook 04 - LangGraph e RAG Medico

## Tech Challenge Fase 3 - Assistente Virtual Medico

Este notebook cobre:
1. Conceitos do LangGraph (grafos de decisao)
2. Implementacao do grafo medico com LangGraph
3. RAG (Retrieval-Augmented Generation) para consultas medicas
4. Integracao do RAG com LangGraph

---
## 1. Configuracao do Ambiente

In [1]:
import os
import sys
import torch
from dotenv import load_dotenv
from typing import TypedDict, List, Optional
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

load_dotenv()

# Configuracao do Modelo Fine-Tuned
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "../models/assistente_medico_final"

# Modulos centralizados
sys.path.append('..')
from src.model_loader import load_llm
from src.langgraph_nodes import (
    MedicoState,
    classificar_consulta,
    recuperar_documentos_simulado,
    filtrar_relevancia,
    validar_qualidade,
    gerar_resposta,
    verificar_alerta,
    criar_grafo_basico,
    criar_grafo_rag,
)

print("Ambiente configurado!")
print(f"Modelo base: {MODEL_NAME}")
print(f"Adapter: {ADAPTER_PATH}")

/Users/rodrigofranco/Environment/FIAP/fase 3/aulas/projeto-assistente-medico/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ambiente configurado!
Modelo base: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Adapter: ../models/assistente_medico_final


---
## 2. Conceitos do LangGraph

### O que e LangGraph?

O LangGraph e uma biblioteca que organiza fluxos de LLMs como **grafos direcionados** em vez de pipelines lineares.

### Componentes Principais:
- **StateGraph**: Define a estrutura do grafo com TypedDict como esquema de estado
- **Nos (Nodes)**: Funcoes puras que recebem estado e retornam modificacoes
- **Arestas (Edges)**: Transicoes entre nos, incluindo condicionais
- **Estado Compartilhado**: Dicionario tipado que viaja entre os nos

### Por que usar LangGraph?
- Fluxos mais complexos que lineares
- Condicoes e ramificacoes
- Loops e iteracoes
- Paralelismo
- Controle fino do fluxo

---
## 3. Implementacao do Grafo Medico com LangGraph

### 3.1 Definicao do Estado

In [2]:
from src.langgraph_nodes import MedicoState

print("Estado do grafo definido!")
print(f"Campos: {list(MedicoState.__annotations__.keys())}")

Estado do grafo definido!
Campos: ['pergunta', 'classificacao', 'documentos_brutos', 'documentos_filtrados', 'contexto', 'resposta', 'fontes', 'confianca', 'etapa_atual', 'historico_decisoes', 'alerta_necessario', 'mensagem_alerta']


### 3.2 Implementacao dos Nos

In [3]:
# FLAG: True = usar modelo fine-tuned, False = usar modelo base puro
USE_FINETUNED = True

# Carregar LLM usando modulo centralizado
llm = load_llm(
    model_name=MODEL_NAME,
    adapter_path=ADAPTER_PATH,
    use_finetuned=USE_FINETUNED
)

print(f"Modelo carregado com sucesso!")
print(f"Modo: {'Fine-tuned' if USE_FINETUNED else 'Base'}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Carregando modelo base...


Loading weights: 100%|██████████| 201/201 [00:01<00:00, 101.56it/s]


Aplicando adapter LoRA...


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_k', 'max_length', 'pad_token_id', 'temperature', 'repetition_penalty', 'do_sample', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Modelo carregado com sucesso!
Modo: Fine-tuned


In [4]:
# ============================================
# NOS DO GRAFO (importados de src/langgraph_nodes.py)
# ============================================
print("Nos do grafo importados do modulo centralizado!")
print("Nos: classificar_consulta, recuperar_documentos_simulado, filtrar_relevancia, validar_qualidade, gerar_resposta, verificar_alerta")

Nos do grafo importados do modulo centralizado!
Nos: classificar_consulta, recuperar_documentos_simulado, filtrar_relevancia, validar_qualidade, gerar_resposta, verificar_alerta


### 3.3 Construcao do Grafo

In [5]:
# ============================================
# CONSTRUCAO DO GRAFO LANGGRAPH
# ============================================
# Usar funcao centralizada para criar o grafo basico
app = criar_grafo_basico(llm)

print("Grafo compilado com sucesso!")

Grafo compilado com sucesso!


### 3.4 Visualizacao do Grafo

In [6]:
# Visualizar estrutura do grafo
print("Estrutura do grafo:")
try:
    print(app.get_graph().draw_ascii())
except ImportError:
    print("Para visualizar o grafo, instale: pip install grandalf")
    print("\nNos do grafo:")
    for node in app.get_graph().nodes:
        print(f"  - {node}")

Estrutura do grafo:
          +-----------+       
          | __start__ |       
          +-----------+       
                 *            
                 *            
                 *            
          +-------------+     
          | classificar |     
          +-------------+     
                 *            
                 *            
                 *            
          +-----------+       
          | recuperar |       
          +-----------+       
          ***         ..      
         *              ..    
       **                 ..  
+---------+                 . 
| filtrar |               ..  
+---------+             ..    
          ***         ..      
             *      ..        
              **   .          
            +---------+       
            | validar |       
            +---------+       
                 .            
                 .            
                 .            
          +-----------+       
          | respond

### 3.5 Execucao do Grafo

In [7]:
# Executar o grafo com uma pergunta de teste
print("="*60)
print("EXECUCAO DO GRAFO LANGGRAPH")
print("="*60)

entrada = {
    "pergunta": "Qual o protocolo para pneumonia hospitalar em pacientes idosos?",
    "historico_decisoes": []
}

resultado = app.invoke(entrada)

print(f"\nClassificacao: {resultado['classificacao']}")
print(f"Confianca: {resultado['confianca']:.2%}")
print(f"Fontes: {resultado['fontes']}")
print(f"Alerta necessario: {resultado['alerta_necessario']}")
if resultado['alerta_necessario']:
    print(f"Mensagem alerta: {resultado['mensagem_alerta']}")
print(f"\nHistorico de decisoes:")
for i, decisao in enumerate(resultado['historico_decisoes'], 1):
    print(f"  {i}. {decisao}")
print(f"\n{'='*60}")
print("RESPOSTA DO ASSISTENTE:")
print("="*60)
print(resultado['resposta'])

EXECUCAO DO GRAFO LANGGRAPH


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.



Classificacao: ### Instruction:
Classifique a seguinte consulta medica em UMA categoria:
- CLINICA_GERAL
- CARDIOLOGIA
- PNEUMOLOGIA
- NEUROLOGIA
- URGENCIA
- ENFERMAGEM

Consulta: Qual o protocolo para pneumonia hospitalar em pacientes idosos?

Responda APENAS com o nome da categoria.

### Response:
Protocolo de saude: Pneumonia Hospitalar em Pacientes Idosos

1. Fatores de risco:
   - Avasao de peso
   - Diabetes Mellitus
   - Hipertensao arterial
   - Obesidade
   - Tuberculose
   - Infeccao por bacteroides
   - Embarazo e gravidez

2. Protocolo de saude:
   - Diagnosticacao:
      - Exame respiratorio (cavidade esofageal, broncopulmonar)
      - Exame urinario
      - Exame fisiologico (tromsofarmacologia, hematologia, hemodilatacao)
      - Exame cardiovascular
      - Exame neurologico
      - Exame gastroenterologico

   - Tratamento:
      - Antibioticos (pneumococo, Streptococo)
      - Antivirales (ganciclovir, valenciclovir)
      - Antifungicos (amphoterico, fluconazol)
  

---
## 4. RAG (Retrieval-Augmented Generation)

### 4.1 Conceito

RAG combina:
- **Retrieval** (Recuperacao): Busca de documentos relevantes em uma base de conhecimento
- **Augmented** (Aumentado): Enriquece o prompt do LLM com os documentos recuperados
- **Generation** (Geracao): O LLM gera resposta baseada no contexto recuperado

### Vantagens do RAG:
- Reduz alucinacoes do LLM
- Permite acesso a informacoes atualizadas
- Fontes rastreaveis e auditaveis
- Conhecimento especifico do dominio

### 4.2 Vector Store com FAISS

In [8]:
# Importar modulo RAG do projeto
import sys
sys.path.append('..')
from src.rag_module import (
    SimpleDeterministicEmbeddings,
    create_medical_documents,
    create_vector_store,
    create_retriever
)

# Criar documentos medicos
documentos_medicos = create_medical_documents()
print(f"{len(documentos_medicos)} documentos medicos carregados")

# Criar vector store
print("Criando vector store com FAISS...")
vectorstore = create_vector_store(documentos_medicos)
print(f"Vector store criado com {vectorstore.index.ntotal} vetores")

# Criar retriever
retriever = create_retriever(vectorstore, search_k=2)
print("Retriever configurado!")

8 documentos medicos carregados
Criando vector store com FAISS...
Documentos divididos em 8 chunks
Vector store criado com 8 vetores
Vector store criado com 8 vetores
Retriever configurado!


### 4.3 Integracao RAG com LangGraph

In [9]:
# Usar funcao centralizada para criar o grafo com RAG
app_rag = criar_grafo_rag(llm, retriever)

print("Grafo RAG compilado com sucesso!")

Grafo RAG compilado com sucesso!


In [10]:
# Testar o grafo RAG
print("="*60)
print("TESTE DO GRAFO RAG")
print("="*60)

entrada_rag = {
    "pergunta": "Qual o tratamento para sepse em pacientes criticos?",
    "historico_decisoes": []
}

resultado_rag = app_rag.invoke(entrada_rag)

print(f"\nClassificacao: {resultado_rag['classificacao']}")
print(f"Confianca: {resultado_rag['confianca']:.2%}")
print(f"Fontes: {resultado_rag['fontes']}")
print(f"\nHistorico:")
for i, decisao in enumerate(resultado_rag['historico_decisoes'], 1):
    print(f"  {i}. {decisao}")
print(f"\n{'='*60}")
print("RESPOSTA:")
print("="*60)
print(resultado_rag['resposta'])

TESTE DO GRAFO RAG

Classificacao: ### Instruction:
Classifique a seguinte consulta medica em UMA categoria:
- CLINICA_GERAL
- CARDIOLOGIA
- PNEUMOLOGIA
- NEUROLOGIA
- URGENCIA
- ENFERMAGEM

Consulta: Qual o tratamento para sepse em pacientes criticos?

Responda APENAS com o nome da categoria.

### Response:
CATEGORIA: PNEUMOLOGIA

TRATAMENTO:
- Terapia de ventilação intensiva (TIV)
- Atenção primária
- Atenção secundária

CATEGORIA: CARDIOLOGIA

TRATAMENTO:
- Terapia de ventilação intensiva (TIV)
- Atenção primária
- Atenção secundária

CATEGORIA: NEUROLOGIA

TRATAMENTO:
- Terapia de ventilação intensiva (TIV)
- Atenção primária
- Atenção secundária

CATEGORIA: URGENCIA

TRATAMENTO:
- Terapia de ventilação intensiva (TIV)
- Atenção primária
- Atenção secundária
Confianca: 80.00%
Fontes: ['PROTO-002', 'PROTO-007']

Historico:
  1. Classificacao: ### Instruction:
Classifique a seguinte consulta medica em UMA categoria:
- CLINICA_GERAL
- CARDIOLOGIA
- PNEUMOLOGIA
- NEUROLOGIA
- URGENCIA


---
## 5. Padrao ReAct (Reasoning + Acting)

O padrao ReAct e utilizado em agentes que combinam raciocinio com acoes.

In [11]:
# Exemplo de padrao ReAct
def agente_react_medico(pergunta: str) -> str:
    """
    Agente medico seguindo o padrao ReAct.
    """
    # Thought: Raciocinio
    thought = f"""
    Thought: Preciso analisar a pergunta medica:
    1. Identificar a condicao clinica
    2. Buscar protocolo relevante
    3. Verificar se ha contraindicacoes
    4. Gerar resposta segura
    """
    
    # Action: Buscar protocolo
    documentos_encontrados = retriever.invoke(pergunta)
    
    # Observation: Processar resultados
    if documentos_encontrados:
        obs = f"Observation: Encontrei {len(documentos_encontrados)} protocolos relevantes."
        contexto = "\n".join([doc.page_content for doc in documentos_encontrados])
    else:
        obs = "Observation: Nenhum protocolo especifico encontrado. Usando conhecimento geral."
        contexto = ""
    
    # Gerar resposta final
    prompt_final = f"""### Instruction:
Pergunta: {pergunta}
Contexto: {contexto}

Gere uma resposta medica segura e estruturada.
Inclua aviso de seguranca.

### Response:
"""
    
    resposta = llm.invoke(prompt_final)
    
    return f"{thought}\n{obs}\n\nResposta:\n{resposta}"

# Testar agente ReAct
print("="*60)
print("AGENTE REACT MEDICO")
print("="*60)
resultado_react = agente_react_medico("Paciente com insuficiencia cardiaca aguda. Qual conduta?")
print(resultado_react)

AGENTE REACT MEDICO

    Thought: Preciso analisar a pergunta medica:
    1. Identificar a condicao clinica
    2. Buscar protocolo relevante
    3. Verificar se ha contraindicacoes
    4. Gerar resposta segura
    
Observation: Encontrei 2 protocolos relevantes.

Resposta:
### Instruction:
Pergunta: Paciente com insuficiencia cardiaca aguda. Qual conduta?
Contexto: Protocolo de Dor Lombar Aguda:
Condutas:
1. Manter atividade (evitar repouso)
2. Dipirona 1g 6/6h ou Paracetamol 750mg 6/6h
3. Ibuprofeno 600mg 8/8h
4. Fisioterapia e alongamento
5. Avaliar sinais de alerta (red flags)
6. Encaminhar se deficit neurologico
Protocolo de Diabetes Mellitus Tipo 2:
Condutas:
1. HbA1c meta < 7%
2. Metformina 500-2000mg/dia
3. Dieta hipocalorica se sobrepeso
4. Exercicio 150min/semana
5. Monitoramento glicemico regular
6. Rastreamento de complicacoes anuais

Gere uma resposta medica segura e estruturada.
Inclua aviso de seguranca.

### Response:
This patient's condition is aggravated by a history 

---
## 6. Resumo

### Componentes Implementados:

| Componente | Funcao |
|------------|--------|
| **StateGraph** | Estrutura do grafo |
| **TypedDict** | Estado compartilhado |
| **Nos** | classificar, recuperar, filtrar, validar, responder, alertar |
| **Arestas Condicionais** | Fluxo baseado em validacao |
| **FAISS** | Vector store para busca semantica |
| **RAG** | Recuperacao + Geracao |
| **ReAct** | Padrao Reasoning + Acting |

### Fluxo do Assistente Medico:
```
Pergunta -> Classificacao -> Recuperacao (RAG) -> Filtragem -> 
Validacao -> (se aprovado) Resposta -> Alerta -> Fim
                     (se reprocessar) ^
```

### Proximo notebook:
O notebook `05_sistema_completo.ipynb` integrara tudo com seguranca e logging detalhado.

In [12]:
print("\n=== NOTEBOOK 04 CONCLUIDO ===")
print("LangGraph e RAG implementados com sucesso!")


=== NOTEBOOK 04 CONCLUIDO ===
LangGraph e RAG implementados com sucesso!
